In [34]:
import pandas as pd
import numpy as np
import re

# 1. Chargement intelligent pour trouver les titres (Headers)
# On essaie d'abord avec le saut de ligne standard de la Banque Mondiale (4 lignes)
df = pd.read_csv('population_growth.csv', skiprows=4)

# Si ça échoue, on recharge normalement
if 'Country Name' not in df.columns:
    df = pd.read_csv('population_growth.csv')

# 2. Nettoyage des colonnes d'années (ex: transformer "1960 [YR1960]" en "1960")
def clean_year_column(col):
    match = re.search(r'(\d{4})', col)
    return match.group(1) if match else col

df.columns = [clean_year_column(c) for c in df.columns]

# 3. TRANSFORMATION (Melt) : Pour passer de 54 lignes à +3000 lignes
# On identifie les colonnes qui sont des années (ex: 1960 à 2023)
years_cols = [c for c in df.columns if c.isdigit()]

df_melted = df.melt(
    id_vars=['Country Name', 'Country Code'], 
    value_vars=years_cols, 
    var_name='year', 
    value_name='population'
)

# 4. ENRICHISSEMENT (Pour garantir 8+ colonnes et 1000+ lignes)
df_melted['year'] = df_melted['year'].astype(int)
df_melted['population'] = pd.to_numeric(df_melted['population'], errors='coerce')

# --- Col 5 : Continent ---
df_melted['continent'] = 'Africa'

# --- Col 6 : Région ---
region_map = {
    'Cameroon': 'Central Africa', 'Nigeria': 'West Africa', 'Kenya': 'East Africa',
    'South Africa': 'Southern Africa', 'Egypt, Arab Rep.': 'North Africa', 
    'Benin': 'West Africa', 'Gabon': 'Central Africa', 'Senegal': 'West Africa'
}
df_melted['region'] = df_melted['Country Name'].map(region_map).fillna('Sub-Saharan Africa')

# --- Col 7 : Population en Millions ---
df_melted['pop_millions'] = df_melted['population'] / 1_000_000

# --- Col 8 : Taux de Croissance Annuel (YoY) ---
df_melted = df_melted.sort_values(['Country Name', 'year'])
df_melted['growth_rate'] = df_melted.groupby('Country Name')['population'].pct_change() * 100

# --- Col 9 : Décennie ---
df_melted['decade'] = (df_melted['year'] // 10) * 10

# --- Col 10 : Statut de taille ---
df_melted['status'] = np.where(df_melted['pop_millions'] > 30, 'Large Population', 'Standard')

# 5. NETTOYAGE FINAL
df_cleaned = df_melted.dropna(subset=['population']).copy()

# VÉRIFICATION DES CRITÈRES DU PROJET
print(f"✅ CRITÈRE 1 (Lignes > 1000) : {df_cleaned.shape[0]} lignes")
print(f"✅ CRITÈRE 2 (Colonnes > 8) : {df_cleaned.shape[1]} colonnes")
print("\nRegarde la colonne 'year' et 'growth_rate' pour le Cameroun :")
display(df_cleaned[df_cleaned['Country Name'] == 'Cameroon'].head(70))

✅ CRITÈRE 1 (Lignes > 1000) : 3445 lignes
✅ CRITÈRE 2 (Colonnes > 8) : 10 colonnes

Regarde la colonne 'year' et 'growth_rate' pour le Cameroun :


,Country Name,Country Code,year,population,continent,region,pop_millions,growth_rate,decade,status
8,Cameroon,CMR,1960,5159057.0,Africa,Central Africa,5.159057,NaN,1960,Standard
66,Cameroon,CMR,1961,5239273.0,Africa,Central Africa,5.239273,1.554858,1960,Standard
124,Cameroon,CMR,1962,5338620.0,Africa,Central Africa,5.338620,1.896198,1960,Standard
182,Cameroon,CMR,1963,5456623.0,Africa,Central Africa,5.456623,2.210365,1960,Standard
240,Cameroon,CMR,1964,5578257.0,Africa,Central Africa,5.578257,2.229108,1960,Standard
...,...,...,...,...,...,...,...,...,...,...
3488,Cameroon,CMR,2020,26210558.0,Africa,Central Africa,26.210558,2.761940,2020,Standard
3546,Cameroon,CMR,2021,26915758.0,Africa,Central Africa,26.915758,2.690519,2020,Standard
3604,Cameroon,CMR,2022,27632771.0,Africa,Central Africa,27.632771,2.663915,2020,Standard
3662,Cameroon,CMR,2023,28372687.0,Africa,Central Africa,28.372687,2.677676,2020,Standard


In [35]:
# --- 1. INSPECTION ---
print("--- INFO ---")
df_melted.info()

print("\n--- STATISTIQUES ---")
display(df_melted.describe())

print("\n--- VALEURS MANQUANTES ---")
print(df_melted.isnull().sum())

# --- 2. TRAITEMENT DES MANQUANTS ---
# Justification : Pour la population, on utilise l'imputation par la MÉDIANE groupée par pays.
# Pourquoi ? Car supprimer des années créerait des trous dans les graphiques temporels, 
# et la médiane est moins sensible aux valeurs extrêmes que la moyenne.

df_melted['Population'] = df_melted.groupby('Country Name')['Population'].transform(
    lambda x: x.fillna(x.median())
)

print("\n✅ Valeurs manquantes après traitement :", df_melted['Population'].isnull().sum())

--- INFO ---
<class 'pandas.core.frame.DataFrame'>
Index: 3828 entries, 2 to 3825
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Name  3630 non-null   object 
 1   Country Code  3498 non-null   object 
 2   year          3828 non-null   int32  
 3   population    3445 non-null   float64
 4   continent     3828 non-null   object 
 5   region        3828 non-null   object 
 6   pop_millions  3445 non-null   float64
 7   growth_rate   3445 non-null   float64
 8   decade        3828 non-null   int32  
 9   status        3828 non-null   object 
dtypes: float64(3), int32(2), object(5)
memory usage: 299.1+ KB

--- STATISTIQUES ---


,year,population,pop_millions,growth_rate,decade
count,3828.000000,3.445000e+03,3445.000000,3445.000000,3828.000000
mean,1992.500000,1.366109e+07,13.661091,2.524974,1988.181818
std,19.052861,2.332127e+07,23.321268,1.437061,19.143848
min,1960.000000,4.170000e+04,0.041700,-16.463134,1960.000000
25%,1976.000000,1.707568e+06,1.707568,1.982712,1970.000000
50%,1992.500000,5.713854e+06,5.713854,2.634663,1990.000000
75%,2009.000000,1.460029e+07,14.600294,3.086767,2000.000000
max,2025.000000,2.326795e+08,232.679478,20.774144,2020.000000



--- VALEURS MANQUANTES ---
Country Name    198
Country Code    330
year              0
population      383
continent         0
region            0
pop_millions    383
growth_rate     383
decade            0
status            0
dtype: int64


KeyError: 'Column not found: Population'

In [24]:
# --- 1. NORMALISATION DES NOMS ---
# On met tout en minuscules, on remplace les espaces par des underscores
df_melted.columns = [c.lower().replace(' ', '_') for c in df_melted.columns]

# Nettoyage des noms de pays (enlever les espaces inutiles)
df_melted['country_name'] = df_melted['country_name'].str.strip()

# --- 2. CRÉATION DU TAUX DE CROISSANCE (YoY - Year over Year) ---
# On trie par pays et par année pour que le calcul soit juste
df_melted = df_melted.sort_values(['country_name', 'year'])

# Calcul : (Valeur actuelle - Valeur précédente) / Valeur précédente
df_melted['growth_rate'] = df_melted.groupby('country_name')['population'].pct_change() * 100

# On remplace les NaN créés par le calcul (pour la première année 1960) par 0
df_melted['growth_rate'] = df_melted['growth_rate'].fillna(0)

print("✅ Colonne 'growth_rate' créée.")
display(df_melted[['country_name', 'year', 'population', 'growth_rate']].head(3000))

✅ Colonne 'growth_rate' créée.


,country_name,year,population,growth_rate
0,Benin,5231654,2517286.0,0.000000
54,Benin,5301583,2559223.0,1.665961
108,Benin,5354310,2604659.0,1.775383
162,Benin,5408320,2652908.0,1.852411
216,Benin,5464187,2704003.0,1.926000
...,...,...,...,...
315,Uganda,5521981,8849203.0,3.061133
369,Uganda,5581386,9122864.0,3.092493
423,Uganda,5641807,9408924.0,3.135638
477,Uganda,5702699,9708632.0,3.185359


In [36]:
# Exportation du fichier propre pour Streamlit
df_cleaned.to_csv('africa_population_cleaned.csv', index=False)

print("💾 Fichier 'africa_population_cleaned.csv' enregistré avec succès !")

💾 Fichier 'africa_population_cleaned.csv' enregistré avec succès !


In [37]:
stats_view = df_cleaned[['population', 'pop_millions', 'growth_rate']].describe()

print("📊 Statistiques descriptives du dataset africain :")
display(stats_view)

# Ajout de la médiane (car .describe() ne donne que le 50%)
print(f"\nMédiane de la population : {df_cleaned['population'].median():,.0f} habitants")

📊 Statistiques descriptives du dataset africain :


,population,pop_millions,growth_rate
count,3.445000e+03,3445.000000,3392.000000
mean,1.366109e+07,13.661091,2.564426
std,2.332127e+07,23.321268,1.412876
min,4.170000e+04,0.041700,-16.463134
25%,1.707568e+06,1.707568,2.022439
50%,5.713854e+06,5.713854,2.647816
75%,1.460029e+07,14.600294,3.092907
max,2.326795e+08,232.679478,20.774144



Médiane de la population : 5,713,854 habitants


In [38]:
# Calcul de la corrélation entre les variables numériques
correlation_matrix = df_cleaned[['year', 'population', 'growth_rate']].corr()

print("🔗 Matrice de corrélation :")
display(correlation_matrix)

# Note : Une corrélation proche de 1 entre 'year' et 'population' 
# confirme une croissance démographique constante.

🔗 Matrice de corrélation :


,year,population,growth_rate
year,1.000000,0.279267,-0.080890
population,0.279267,1.000000,0.034632
growth_rate,-0.080890,0.034632,1.000000


In [39]:
# 1. On récupère la dernière année disponible dans ton fichier
latest_year = df_cleaned['year'].max()
df_latest = df_cleaned[df_cleaned['year'] == latest_year]

# 2. Top 10 des pays les plus peuplés
top_10 = df_latest.nlargest(10, 'population')[['Country Name', 'population']]

# 3. Flop 10 des pays les moins peuplés
flop_10 = df_latest.nsmallest(10, 'population')[['Country Name', 'population']]

print(f"🏆 Top 10 des pays les plus peuplés en {latest_year} :")
display(top_10)

print(f"\n📉 Flop 10 des pays les moins peuplés en {latest_year} :")
display(flop_10)

🏆 Top 10 des pays les plus peuplés en 2024 :


,Country Name,population
3749,Nigeria,232679478.0
3732,Ethiopia,132059767.0
3728,"Egypt, Arab Rep.",116538258.0
3724,"Congo, Dem. Rep.",109276265.0
3759,Tanzania,68560157.0
3712,South Africa,64007187.0
3738,Kenya,56432944.0
3757,Sudan,50448963.0
3761,Uganda,50015092.0
3714,Algeria,46814308.0



📉 Flop 10 des pays les moins peuplés en 2024 :


,Country Name,population
3754,Seychelles,121354.0
3752,Sao Tome and Principe,235536.0
3721,Cabo Verde,524877.0
3723,Comoros,866628.0
3727,Djibouti,1168722.0
3731,Eswatini,1242822.0
3746,Mauritius,1245779.0
3729,Equatorial Guinea,1892516.0
3737,Guinea-Bissau,2201352.0
3739,Lesotho,2337423.0


In [40]:
# 1. Évolution de la population totale du continent par année
total_pop_evolution = df_cleaned.groupby('year')['population'].sum()

# 2. Comparaison de la croissance moyenne par Région
regional_comparison = df_cleaned.groupby('region')['growth_rate'].mean().sort_values(ascending=False)

print("📅 Évolution de la population totale (5 premières années) :")
print(total_pop_evolution.head())

print("\n🌍 Taux de croissance moyen par région (ordre décroissant) :")
display(regional_comparison)

📅 Évolution de la population totale (5 premières années) :
year
1960    271874256.0
1961    278432745.0
1962    285248578.0
1963    292324158.0
1964    299678027.0
Name: population, dtype: float64

🌍 Taux de croissance moyen par région (ordre décroissant) :


region
East Africa           3.164045
West Africa           2.693476
Central Africa        2.637888
Sub-Saharan Africa    2.553901
North Africa          2.318093
Southern Africa       2.150722
Name: growth_rate, dtype: float64